# Thick Elliptical Tube Parametric Study

This notebook reproduces and extends the analysis workflow from `Cylinder_example.py`.

It is organized into three studies:

- **Study A:** Aspect ratio sweep with constant uniform thickness.
- **Study B:** Thickness profile sweep at fixed aspect ratios (`AR = 1` and `AR = 2`).
- **Study C:** Material sensitivity using multiple `mater.txt` definitions.

Narrative targets:

1. Curvature is correlated with hoop stress variation.
2. Non-uniform thickness concentrates stress.
3. Material properties shift stress response magnitude.


## Setup Notes

- Run this notebook from the repository root so SolidSpy can find and write `nodes.txt`, `eles.txt`, `loads.txt`, and `mater.txt`.
- This notebook regenerates mesh and load files **for every case** before solving.
- Plane-strain effective properties should be handled consistently with your `mater.txt` convention.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from solidspy import solids_GUI

# Global controls
R_I = 300.0
P_INTERNAL = 100.0
N_THETA = 80
N_R = 15
PHI_GRID = np.linspace(0.0, 180.0, 200)

ROOT = Path.cwd()
MATER_PATH = ROOT / "mater.txt"

# Fixed colors for Study A (AR sweep)
AR_VALUES = [1.0, 1.25, 1.75, 2.0]
AR_COLORS = {
    1.0: "#1f77b4",
    1.25: "#ff7f0e",
    1.75: "#2ca02c",
    2.0: "#d62728",
}

# Fixed colors for Study B (thickness conditions)
THICKNESS_CONDITIONS = {
    "uniform": (20.0, 20.0, 20.0),
    "bottom_thick_side_thin": (20.0, 12.0, 32.0),
    "sides_thick_top_bottom_thin": (12.0, 30.0, 12.0),
    "top_thin_bottom_thick_sides_mid": (10.0, 20.0, 30.0),
}
THICKNESS_COLORS = {
    "uniform": "#1f77b4",
    "bottom_thick_side_thin": "#ff7f0e",
    "sides_thick_top_bottom_thin": "#2ca02c",
    "top_thin_bottom_thick_sides_mid": "#d62728",
}


In [ ]:
def write_mater(E: float, nu: float) -> None:
    """Overwrite mater.txt for the current run."""
    MATER_PATH.write_text(f"{E:.6e} {nu:.6e}\n")


def generate_and_save_model(r_i, AR, t_top, t_side, t_bot, n_t, n_r, pressure):
    """Generate half-tube mesh, BCs, and pressure loads and save SolidSpy files."""
    theta = np.linspace(-np.pi / 2.0, np.pi / 2.0, n_t, endpoint=True)
    xi = np.linspace(0.0, 1.0, n_r)

    a, b = r_i / AR, r_i
    A_t = (2.0 * t_side + t_top + t_bot) / 4.0
    B_t = (t_top - t_bot) / 2.0
    C_t = (2.0 * t_side - t_top - t_bot) / 4.0

    nodes_x, nodes_y = [], []
    for r_frac in xi:
        r_in = (a * b) / np.sqrt((b * np.cos(theta)) ** 2 + (a * np.sin(theta)) ** 2)
        t_theta = A_t + B_t * np.sin(theta) + C_t * np.cos(2.0 * theta)
        r_out = r_in + r_frac * t_theta
        nodes_x.extend(r_out * np.cos(theta))
        nodes_y.extend(r_out * np.sin(theta))

    nodes_x = np.array(nodes_x)
    nodes_y = np.array(nodes_y)

    triangles = []
    for i in range(n_r - 1):
        for j in range(n_t - 1):
            n1, n2 = i * n_t + j, i * n_t + (j + 1)
            n3, n4 = (i + 1) * n_t + j, (i + 1) * n_t + (j + 1)
            triangles.extend([[n1, n3, n2], [n2, n3, n4]])
    triangles = np.array(triangles)

    min_y = np.min(nodes_y)
    with open("nodes.txt", "w", encoding="utf-8") as fnodes:
        for i, (x, y) in enumerate(zip(nodes_x, nodes_y)):
            bc_x = -1 if abs(x) < 1e-6 else 0
            bc_y = -1 if (abs(x) < 1e-6 and abs(y - min_y) < 1e-6) else 0
            fnodes.write(f"{i} {x:.8f} {y:.8f} {bc_x} {bc_y}\n")

    with open("eles.txt", "w", encoding="utf-8") as feles:
        for i, tri in enumerate(triangles):
            feles.write(f"{i} 3 0 {tri[0]} {tri[1]} {tri[2]}\n")

    F_x = np.zeros(len(nodes_x))
    F_y = np.zeros(len(nodes_y))
    for j in range(n_t - 1):
        n1, n2 = j, j + 1
        dx = nodes_x[n2] - nodes_x[n1]
        dy = nodes_y[n2] - nodes_y[n1]

        fx_seg, fy_seg = pressure * dy, pressure * -dx
        F_x[n1] += 0.5 * fx_seg
        F_y[n1] += 0.5 * fy_seg
        F_x[n2] += 0.5 * fx_seg
        F_y[n2] += 0.5 * fy_seg

    with open("loads.txt", "w", encoding="utf-8") as floads:
        for i in range(len(nodes_x)):
            if abs(F_x[i]) > 1e-8 or abs(F_y[i]) > 1e-8:
                floads.write(f"{i} {F_x[i]:.6f} {F_y[i]:.6f}\n")

    return nodes_x, nodes_y


In [ ]:
def run_case(r_i, AR, t_top, t_side, t_bot, pressure, n_theta, n_r, phi_grid, material=None):
    """Run one FEA case and return profiles for plotting."""
    if material is not None:
        write_mater(material[0], material[1])

    X, Y = generate_and_save_model(r_i, AR, t_top, t_side, t_bot, n_theta, n_r, pressure)
    disp, strain, stress = solids_GUI(plot_contours=False, compute_strains=True, folder="./")

    sig_xx = stress[:, 0]
    sig_yy = stress[:, 1]
    tau_xy = stress[:, 2]

    theta_all = np.arctan2(Y, X)
    theta_in_raw = np.arctan2(Y[:n_theta], X[:n_theta])
    sort_idx = np.argsort(theta_in_raw)

    x_in = X[:n_theta][sort_idx]
    y_in = Y[:n_theta][sort_idx]
    theta_in_sorted = theta_in_raw[sort_idx]

    dx_in = np.gradient(x_in)
    dy_in = np.gradient(y_in)
    alpha_inner = np.arctan2(dy_in, dx_in) - (np.pi / 2.0)

    closest_idx = np.argmin(np.abs(theta_all[:, np.newaxis] - theta_in_sorted), axis=1)
    alpha_all = alpha_inner[closest_idx]
    c = np.cos(alpha_all)
    s = np.sin(alpha_all)

    sig_tt = sig_xx * s**2 + sig_yy * c**2 - 2.0 * tau_xy * s * c

    sig_tt_in_raw = sig_tt[:n_theta]
    phi_in_deg_raw = 90.0 - np.degrees(theta_in_raw)
    sort_idx2 = np.argsort(phi_in_deg_raw)
    sig_tt_smooth = np.interp(phi_grid, phi_in_deg_raw[sort_idx2], sig_tt_in_raw[sort_idx2])

    theta_math = np.pi / 2.0 - np.deg2rad(phi_grid)
    A_t = (2.0 * t_side + t_top + t_bot) / 4.0
    B_t = (t_top - t_bot) / 2.0
    C_t = (2.0 * t_side - t_top - t_bot) / 4.0
    thickness = A_t + B_t * np.sin(theta_math) + C_t * np.cos(2.0 * theta_math)

    a = r_i / AR
    b = r_i
    numerator = a * b
    denominator = (a**2 * np.sin(theta_math) ** 2 + b**2 * np.cos(theta_math) ** 2) ** 1.5
    kappa = numerator / denominator

    return {
        "phi": phi_grid,
        "sig_tt": sig_tt_smooth,
        "kappa": kappa,
        "thickness": thickness,
        "AR": AR,
        "t_top": t_top,
        "t_side": t_side,
        "t_bot": t_bot,
    }


## Study A — Aspect Ratio Sweep (Uniform Thickness)

Parameters:

- `AR ∈ {1.0, 1.25, 1.75, 2.0}`
- `t_top = t_side = t_bot = t_ref = 20`
- Fixed pressure, mesh density, and inner radius


In [ ]:
results_A = {}
t_ref = 20.0

for ar in AR_VALUES:
    results_A[ar] = run_case(
        r_i=R_I,
        AR=ar,
        t_top=t_ref,
        t_side=t_ref,
        t_bot=t_ref,
        pressure=P_INTERNAL,
        n_theta=N_THETA,
        n_r=N_R,
        phi_grid=PHI_GRID,
        material=None,
    )

fig, ax = plt.subplots(figsize=(8, 5))
for ar in AR_VALUES:
    r = results_A[ar]
    ax.plot(r["phi"], r["sig_tt"], color=AR_COLORS[ar], lw=2.2, label=f"AR = {ar}")
ax.set_xlabel("Angle from Top, φ (deg)")
ax.set_ylabel("Hoop Stress, σ_tt")
ax.set_title("Study A1: Hoop Stress vs Angle (Uniform Thickness)")
ax.grid(alpha=0.3)
ax.legend()
plt.show()

fig, ax = plt.subplots(figsize=(8, 5))
for ar in AR_VALUES:
    r = results_A[ar]
    ax.plot(r["phi"], r["kappa"], color=AR_COLORS[ar], lw=2.2, label=f"AR = {ar}")
ax.set_xlabel("Angle from Top, φ (deg)")
ax.set_ylabel("Curvature, κ")
ax.set_title("Study A2: Curvature vs Angle (Uniform Thickness)")
ax.grid(alpha=0.3)
ax.legend()
plt.show()


## Study B — Thickness Profile Effects at AR = 1 and AR = 2

Four thickness conditions are compared with fixed color mapping across all plots:

1. `uniform`
2. `bottom_thick_side_thin`
3. `sides_thick_top_bottom_thin`
4. `top_thin_bottom_thick_sides_mid`


In [ ]:
def run_study_B_for_ar(ar_value):
    out = {}
    for name, (t_top, t_side, t_bot) in THICKNESS_CONDITIONS.items():
        out[name] = run_case(
            r_i=R_I,
            AR=ar_value,
            t_top=t_top,
            t_side=t_side,
            t_bot=t_bot,
            pressure=P_INTERNAL,
            n_theta=N_THETA,
            n_r=N_R,
            phi_grid=PHI_GRID,
            material=None,
        )
    return out

for ar_value in [1.0, 2.0]:
    study_B = run_study_B_for_ar(ar_value)

    fig, axs = plt.subplots(1, 3, figsize=(16, 4.5), constrained_layout=True)

    for name, data in study_B.items():
        color = THICKNESS_COLORS[name]
        axs[0].plot(data["phi"], data["sig_tt"], color=color, lw=2.0, label=name)
        axs[1].plot(data["phi"], data["kappa"], color=color, lw=2.0, label=name)
        axs[2].plot(data["phi"], data["thickness"], color=color, lw=2.0, label=name)

    axs[0].set_title(f"AR={ar_value}: σ_tt(φ)")
    axs[1].set_title(f"AR={ar_value}: κ(φ)")
    axs[2].set_title(f"AR={ar_value}: t(φ)")

    axs[0].set_ylabel("Value")
    for ax in axs:
        ax.set_xlabel("Angle from Top, φ (deg)")
        ax.grid(alpha=0.3)

    handles, labels = axs[2].get_legend_handles_labels()
    fig.legend(handles, labels, ncol=2, loc="lower center", bbox_to_anchor=(0.5, -0.08))
    plt.show()

# Thickness-only comparison panel (optional visualization)
fig, ax = plt.subplots(figsize=(8, 5))
for name, (t_top, t_side, t_bot) in THICKNESS_CONDITIONS.items():
    theta_math = np.pi / 2.0 - np.deg2rad(PHI_GRID)
    A_t = (2.0 * t_side + t_top + t_bot) / 4.0
    B_t = (t_top - t_bot) / 2.0
    C_t = (2.0 * t_side - t_top - t_bot) / 4.0
    thickness = A_t + B_t * np.sin(theta_math) + C_t * np.cos(2.0 * theta_math)
    ax.plot(PHI_GRID, thickness, color=THICKNESS_COLORS[name], lw=2.0, label=name)
ax.set_xlabel("Angle from Top, φ (deg)")
ax.set_ylabel("Thickness, t")
ax.set_title("Thickness Profiles Used in Study B")
ax.grid(alpha=0.3)
ax.legend()
plt.show()


## Study C — Material Sensitivity

This section compares at least two material definitions by rewriting `mater.txt` before each solve.

Representative geometry: `AR = 1`, uniform thickness.


In [ ]:
# Preserve current mater.txt so the notebook can restore it
original_mater = MATER_PATH.read_text() if MATER_PATH.exists() else None

materials = {
    "baseline_E2e11_nu0.30": (2.0e11, 0.30),
    "softer_E1e11_nu0.30": (1.0e11, 0.30),
}
material_colors = {
    "baseline_E2e11_nu0.30": "#1f77b4",
    "softer_E1e11_nu0.30": "#d62728",
}

results_C = {}
for name, mat in materials.items():
    results_C[name] = run_case(
        r_i=R_I,
        AR=1.0,
        t_top=20.0,
        t_side=20.0,
        t_bot=20.0,
        pressure=P_INTERNAL,
        n_theta=N_THETA,
        n_r=N_R,
        phi_grid=PHI_GRID,
        material=mat,
    )

fig, ax = plt.subplots(figsize=(8, 5))
for name, data in results_C.items():
    ax.plot(data["phi"], data["sig_tt"], color=material_colors[name], lw=2.2, label=name)
ax.set_xlabel("Angle from Top, φ (deg)")
ax.set_ylabel("Hoop Stress, σ_tt")
ax.set_title("Study C: Material Sensitivity of Hoop Stress")
ax.grid(alpha=0.3)
ax.legend()
plt.show()

# Restore original material file
if original_mater is not None:
    MATER_PATH.write_text(original_mater)


## Interpretation Notes

- **Curvature-driven response:** In Study A, compare how `κ(φ)` and `σ_tt(φ)` evolve as AR changes. Regions with higher curvature should align with stronger stress variation trends.
- **Thickness concentration effects:** In Study B, stress redistribution across conditions follows the imposed `t(φ)` profile; thinner regions tend to carry higher stress.
- **Material scaling:** In Study C, changing `mater.txt` shifts stress response magnitude, demonstrating constitutive dependence under fixed geometry/loading.
